In [ ]:
import os
import glob

import numpy as np
import rasterio
from rasterio.merge import merge
from scipy import ndimage

# Clean classified sam3 tiles, removing edge pixels and creating a mosaic

INPUT_FOLDER = r"D:\mwalter\sam\kigali_high_density_8_31_512_house_overlap_256"
OUTPUT_FOLDER = r"D:\mwalter\sam\kigali_high_density_8_31_512_house_overlap_256_cleaned"

# Temporary cleaned tiles
CLEANED_FOLDER = os.path.join(
    INPUT_FOLDER,
    "edge_removed"
)

# Final mosaicked raster
OUTPUT_MOSAIC = os.path.join(
    INPUT_FOLDER,
    "SAM3_buildings_mosaic.tif"
)

# Your tile configuration
TILE_SIZE = 512
OVERLAP = 256

# Pixels within this distance of the tile edge are considered part of the edge.

# 1 = only objects actually touching the outermost pixel
# 5 = objects entering the outer 5 pixels
# 10 = objects entering the outer 10 pixels
EDGE_WIDTH = 10

# Remove extremely small detections
MIN_BUILDING_PIXELS = 10


os.makedirs(CLEANED_FOLDER, exist_ok=True)


def remove_edge_buildings(input_file, output_file):

    print(f"Reading: {os.path.basename(input_file)}")

    with rasterio.open(input_file) as src:

        data = src.read(1)

        profile = src.profile.copy()

        height = src.height
        width = src.width

# Get connected pixels

    structure = np.ones(
        (3, 3),
        dtype=np.uint8
    )

    labeled, num_objects = ndimage.label(
        building_pixels,
        structure=structure
    )

    # Find those touching boundary

    edge_labels = set()

    # Top
    if EDGE_WIDTH > 0:
        edge_labels.update(
            np.unique(
                labeled[
                    0:EDGE_WIDTH,
                    :
                ]
            )
        )

        # Bottom
        edge_labels.update(
            np.unique(
                labeled[
                    height - EDGE_WIDTH:height,
                    :
                ]
            )
        )

        # Left
        edge_labels.update(
            np.unique(
                labeled[
                    :,
                    0:EDGE_WIDTH
                ]
            )
        )

        # Right
        edge_labels.update(
            np.unique(
                labeled[
                    :,
                    width - EDGE_WIDTH:width
                ]
            )
        )

    # Remove background label
    edge_labels.discard(0)

    # Remove edge pixels

    cleaned = data.copy()

    if edge_labels:

        # Create a mask containing all edge objects
        edge_object_mask = np.isin(
            labeled,
            list(edge_labels)
        )

        cleaned[edge_object_mask] = 0

    # Remove small objects

    if MIN_BUILDING_PIXELS > 0:

        cleaned_binary = cleaned > 0

        labeled_clean, num_clean = ndimage.label(
            cleaned_binary,
            structure=structure
        )

        # Number of pixels in each object
        object_sizes = np.bincount(
            labeled_clean.ravel()
        )

        # Identify small objects
        small_objects = (
            object_sizes < MIN_BUILDING_PIXELS
        )

        # Never remove background
        small_objects[0] = False

        small_mask = small_objects[
            labeled_clean
        ]

        cleaned[small_mask] = 0


    profile.update(
        count=1,
        nodata=0,
        compress="lzw"
    )


    with rasterio.open(
        output_file,
        "w",
        **profile
    ) as dst:

        dst.write(
            cleaned,
            1
        )

    return np.count_nonzero(cleaned)


# Get tiles

input_files = sorted(
    glob.glob(
        os.path.join(
            INPUT_FOLDER,
            "*.tif"
        )
    )
)

# Also look for .TIF
input_files += sorted(
    glob.glob(
        os.path.join(
            INPUT_FOLDER,
            "*.TIF"
        )
    )
)

# Remove duplicates
input_files = list(dict.fromkeys(input_files))


if not input_files:
    raise RuntimeError(
        "No .tif files were found in the input folder"
    )

# Process tiles

cleaned_files = []

for i, input_file in enumerate(
    input_files,
    start=1
):

    filename = os.path.basename(
        input_file
    )

    output_file = os.path.join(
        CLEANED_FOLDER,
        filename
    )

    pixels_kept = remove_edge_buildings(
        input_file,
        output_file
    )


    cleaned_files.append(
        output_file
    )


src_files = []

try:
    # Open cleaned tiles
    for file in cleaned_files:

        src = rasterio.open(file)

        src_files.append(src)

# Merge using first

    mosaic, mosaic_transform = merge(
        src_files,
        method="first",
        nodata=0
    )

# Get metadata
    out_meta = src_files[0].meta.copy()

    out_meta.update({
        "driver": "GTiff",
        "height": mosaic.shape[1],
        "width": mosaic.shape[2],
        "transform": mosaic_transform,
        "count": 1,
        "nodata": 0,
        "compress": "lzw",
        "BIGTIFF": "IF_SAFER"})


    with rasterio.open(
        OUTPUT_MOSAIC,
        "w",
        **out_meta
    ) as dest:

        dest.write(
            mosaic[0],
            1
        )

finally:

    for src in src_files:
        src.close()




SAM3 EDGE DETECTION REMOVAL
Input folder:   D:\mwalter\sam\kigali_high_density_8_31_512_house_overlap_256
Tile size:      512 x 512
Overlap:        256 pixels
Edge width:     10 pixels
Minimum pixels: 10
Number of tiles: 332


[1/332] mask_kigali_high_density_8_280.TIF
Reading: mask_kigali_high_density_8_280.TIF
  Found 32 connected objects
  Removing 7 edge objects
  Building pixels retained: 25,618

[2/332] mask_kigali_high_density_8_281.TIF
Reading: mask_kigali_high_density_8_281.TIF
  Found 49 connected objects
  Removing 12 edge objects
  Building pixels retained: 21,625

[3/332] mask_kigali_high_density_8_2810.TIF
Reading: mask_kigali_high_density_8_2810.TIF
  Found 38 connected objects
  Removing 12 edge objects
  Building pixels retained: 74,220

[4/332] mask_kigali_high_density_8_28100.TIF
Reading: mask_kigali_high_density_8_28100.TIF
  Found 41 connected objects
  Removing 21 edge objects
  Building pixels retained: 44,316

[5/332] mask_kigali_high_density_8_28101.TIF
Readin